# LeetCode 835 — Image Overlap bằng Python

Link đề: https://leetcode.com/problems/image-overlap/ · Độ khó: **Medium** · Chủ đề: Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def largestOverlap(self, img1: List[List[int]], img2: List[List[int]]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên). Notebook có oracle "dịch ảnh thật sự" (tạo ma trận đã dịch, ô trượt ra ngoài bị xoá) để đối chiếu mọi cách, kể cả cách dùng **nhân số nguyên lớn** để tính tương quan chéo.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai ảnh nhị phân `n × n`; **tịnh tiến** một ảnh (trượt trái/phải/lên/xuống, **không xoay**, bit trượt ra ngoài bị **xoá**) rồi đặt lên ảnh kia; tính số vị trí mà **cả hai đều là `1`**; tìm giá trị lớn nhất.
2. Nắm các công cụ Python: ma trận list-of-list, `collections.Counter` với khoá là tuple, ghép bit thành số nguyên (`<<`, `>>`, `&`, `int.bit_count()`), nhân số nguyên lớn như nhân đa thức.
3. Đi từ **thử mọi phép dịch rồi đếm** → **đếm vector dịch giữa mọi cặp ô `1`** → **dùng bitmask theo hàng** → **tương quan chéo bằng một phép nhân số nguyên lớn** (kỹ thuật Kronecker).
4. Hiểu mấu chốt: một phép dịch `(dx, dy)` làm ô `1` ở `(i, j)` của ảnh 1 trùng ô `1` ở `(i + dx, j + dy)` của ảnh 2 **khi và chỉ khi** vector từ ô của ảnh 1 tới ô của ảnh 2 bằng `(dx, dy)`. Vậy đáp án là **vector xuất hiện nhiều nhất** trong mọi cặp `(ô 1 của ảnh 1, ô 1 của ảnh 2)`.
5. Tránh các lỗi kinh điển: chỉ thử dịch một chiều, dịch **vòng** (wrap-around) thay vì xoá, đếm cả cặp `0-0`, và nhầm hướng của vector dịch.

## 2. Đề bài & ví dụ

Cho hai ảnh `img1`, `img2` là ma trận nhị phân vuông `n × n` (giá trị `0` hoặc `1`).

Ta **tịnh tiến** một ảnh tuỳ ý: trượt toàn bộ các bit `1` sang trái, phải, lên và/hoặc xuống một số ô tuỳ ý rồi đặt lên ảnh kia. **Độ chồng (overlap)** là số vị trí mà **cả hai** ảnh đều có `1`. Phép tịnh tiến **không** gồm phép xoay; các bit `1` bị đẩy ra ngoài ma trận sẽ **bị xoá**.

Trả về **độ chồng lớn nhất** có thể.

**Ví dụ 1:** `img1 = [[1,1,0],[0,1,0],[0,1,0]]`, `img2 = [[0,0,0],[0,1,1],[0,0,1]]` → `3`. Dịch `img1` sang phải `1` và xuống `1`: ba vị trí chồng.

**Ví dụ 2:** `img1 = [[1]]`, `img2 = [[1]]` → `1`.

**Ví dụ 3:** `img1 = [[0]]`, `img2 = [[0]]` → `0`.

**Ràng buộc:** `n == img1.length == img1[i].length == img2.length == img2[i].length`; `1 <= n <= 30`; các ô là `0` hoặc `1`.

In [1]:
import random
import time
from collections import Counter
from itertools import product
from typing import List

img1 = [[1, 1, 0], [0, 1, 0], [0, 1, 0]]
img2 = [[0, 0, 0], [0, 1, 1], [0, 0, 1]]


def show_pair(a, b, left="img1", right="img2"):
    w = 2 * len(a[0]) - 1
    print(f"{left:<{w}}   {right}")
    for r1, r2 in zip(a, b):
        print(" ".join(".#"[v] for v in r1) + "   " + " ".join(".#"[v] for v in r2))


show_pair(img1, img2)
print("số ô 1:", sum(map(sum, img1)), "và", sum(map(sum, img2)))

img1    img2
# # .   . . .
. # .   . # #
. # .   . . #
số ô 1: 4 và 3


## 3. Kiến thức nền

### 3.1 Dịch ma trận và xoá phần tràn

Dịch ảnh `A` đi `(dx, dy)` nghĩa là ô `1` ở `(i, j)` chuyển tới `(i + dx, j + dy)`; nếu toạ độ mới nằm ngoài `[0, n)` thì ô đó **biến mất**. Ta viết hàm tạo ma trận đã dịch để thấy rõ.

In [2]:
def shifted(a, dx, dy):
    n = len(a)
    out = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if a[i][j]:
                x, y = i + dx, j + dy
                if 0 <= x < n and 0 <= y < n:
                    out[x][y] = 1
    return out


print("dịch (+1, +1):")
show_pair(img1, shifted(img1, 1, 1), "gốc", "dịch (1,1)")
print("dịch (0, +2): các bit 1 trượt ra ngoài bị xoá")
show_pair(img1, shifted(img1, 0, 2), "gốc", "dịch (0,2)")

dịch (+1, +1):
gốc     dịch (1,1)
# # .   . . .
. # .   . # #
. # .   . . #
dịch (0, +2): các bit 1 trượt ra ngoài bị xoá
gốc     dịch (0,2)
# # .   . . #
. # .   . . .
. # .   . . .


### 3.2 Đếm chồng: `AND` từng ô

Độ chồng của hai ma trận bằng số ô mà **cả hai** là `1`: `Σ a[i][j] · b[i][j]`. Chú ý **không** đếm ô mà cả hai là `0` (đó là "khớp" nhưng không phải chồng của bit `1`).

In [3]:
def overlap(a, b):
    n = len(a)
    return sum(a[i][j] & b[i][j] for i in range(n) for j in range(n))


print(overlap(shifted(img1, 1, 1), img2), "| chưa dịch:", overlap(img1, img2))

3 | chưa dịch: 1


### 3.3 `Counter` với khoá là tuple

`Counter` đếm số lần xuất hiện. Khoá có thể là **tuple** như `(dx, dy)`. `max(counter.values(), default=0)` cho tần suất lớn nhất (và `0` nếu rỗng).

In [4]:
c = Counter()
for v in [(1, 1), (0, 0), (1, 1), (-1, 0), (1, 1)]:
    c[v] += 1
print(c, "| nhiều nhất:", c.most_common(1), "| rỗng:", max(Counter().values(), default=0))

Counter({(1, 1): 3, (0, 0): 1, (-1, 0): 1}) | nhiều nhất: [((1, 1), 3)] | rỗng: 0


### 3.4 Bit và số nguyên: ghép một hàng thành một số

Một hàng nhị phân `[1, 0, 1]` có thể là số `0b101 = 5` (bit `j` ↔ cột `j`). Khi đó: `a & b` là AND từng bit cùng lúc; `a << k`/`a >> k` dịch cả hàng; `x.bit_count()` đếm số bit `1`. Dịch trái làm bit tràn ra phải **ngoài `n` bit** — ta xoá bằng `& ((1 << n) - 1)`.

In [5]:
def row_to_int(row):
    return sum(v << j for j, v in enumerate(row))


r = row_to_int([1, 0, 1])
print(f"{r:03b}", r, "| dịch trái 1 (giữ 3 bit):", f"{(r << 1) & 0b111:03b}", "| dịch phải 1:", f"{r >> 1:03b}", "| đếm bit 1:", r.bit_count())
print("AND hai hàng:", f"{row_to_int([1, 1, 0]) & row_to_int([0, 1, 1]):03b}")

101 5 | dịch trái 1 (giữ 3 bit): 010 | dịch phải 1: 010 | đếm bit 1: 2
AND hai hàng: 010


### 3.5 Nhân số nguyên lớn như nhân đa thức (Kronecker)

Số nguyên có thể coi là đa thức theo cơ số `2^k`: `N = Σ c_t · 2^{k·t}` với các hệ số `c_t < 2^k`. Nhân hai số như vậy bằng nhân hai **đa thức** (nếu mọi hệ số tích `< 2^k` thì không có nhớ). Python nhân số nguyên lớn rất nhanh (Karatsuba) nên ta có thể tính **tích chập** của hai mảng bằng **một** phép nhân số nguyên.

In [6]:
def poly_to_int(coeffs, k):
    return sum(c << (k * t) for t, c in enumerate(coeffs))


def int_to_poly(x, k, length):
    mask = (1 << k) - 1
    return [(x >> (k * t)) & mask for t in range(length)]


A, B = [1, 2, 3], [4, 5]                       # (1 + 2x + 3x^2) * (4 + 5x) = 4 + 13x + 22x^2 + 15x^3
k = 8
prod = poly_to_int(A, k) * poly_to_int(B, k)
print(int_to_poly(prod, k, 4), "| đúng:", [4, 13, 22, 15])

[4, 13, 22, 15] | đúng: [4, 13, 22, 15]


## 4. Ý tưởng nền tảng / trực giác

Có hai ảnh, ảnh 1 có các ô `1` ở toạ độ `A = {(i, j)}`, ảnh 2 có `B = {(i', j')}`. Dịch ảnh 1 đi `(dx, dy)` thì ô `(i, j) ∈ A` chồng lên `(i', j') ∈ B` đúng khi `(i', j') − (i, j) = (dx, dy)`. Nghĩa là **mỗi cặp** `(a, b)` với `a ∈ A`, `b ∈ B` **bỏ phiếu** cho vector dịch `b − a`. Phép dịch `v` có độ chồng bằng **số phiếu** của `v`.

Với ví dụ 1:

- `A = {(0,0), (0,1), (1,1), (2,1)}` (ảnh 1), `B = {(1,1), (1,2), (2,2)}` (ảnh 2).
- Mọi vector `b − a`: từ `(0,0)`: `(1,1), (1,2), (2,2)`; từ `(0,1)`: `(1,0), (1,1), (2,1)`; từ `(1,1)`: `(0,0), (0,1), (1,1)`; từ `(2,1)`: `(-1,0), (-1,1), (0,1)`.
- Vector `(1,1)` xuất hiện `3` lần (các cặp từ `(0,0)`, `(0,1)`, `(1,1)`) → độ chồng lớn nhất `3`.

**Ghi chú về phạm vi:** ô trượt ra ngoài bị xoá — nhưng nếu `a + v ∈ B` thì `a + v` nằm trong ma trận rồi, nên "xoá" tự động khớp: ta không cần kiểm tra biên khi đếm phiếu.

In [7]:
A = [(i, j) for i in range(3) for j in range(3) if img1[i][j]]
B = [(i, j) for i in range(3) for j in range(3) if img2[i][j]]
votes = Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B)
print("A =", A)
print("B =", B)
print("12 phiếu:", sorted(votes.items()))
print("vector nhiều phiếu nhất:", votes.most_common(1))

A = [(0, 0), (0, 1), (1, 1), (2, 1)]
B = [(1, 1), (1, 2), (2, 2)]
12 phiếu: [((-1, 0), 1), ((-1, 1), 1), ((0, 0), 1), ((0, 1), 2), ((1, 0), 1), ((1, 1), 3), ((1, 2), 1), ((2, 1), 1), ((2, 2), 1)]
vector nhiều phiếu nhất: [((1, 1), 3)]


## 5. Cách 1 — Brute Force: thử mọi phép dịch và đếm chồng

**Ý tưởng:** các phép dịch có ích có `dx, dy ∈ [−(n−1), n−1]` (dịch từ `n` trở lên làm ảnh ra hẳn ngoài). Với mỗi `(dx, dy)`: tạo ảnh đã dịch (hoặc đếm trực tiếp theo chỉ số) và đếm ô `1` chung. Lấy lớn nhất.

In [8]:
def overlap_brute(img1, img2):
    n = len(img1)
    best = 0
    for dx in range(-(n - 1), n):
        for dy in range(-(n - 1), n):
            cnt = overlap(shifted(img1, dx, dy), img2)          # dịch ảnh thật sự rồi đếm
            if cnt > best:
                best = cnt
    return best


print(overlap_brute(img1, img2), overlap_brute([[1]], [[1]]), overlap_brute([[0]], [[0]]))
assert overlap_brute(img1, img2) == 3 and overlap_brute([[1]], [[1]]) == 1 and overlap_brute([[0]], [[0]]) == 0

3 1 0


**Độ phức tạp:** `(2n−1)²` phép dịch × `O(n²)` mỗi lần = `O(n⁴)`; `n = 30` là `3481 × 900 ≈ 3·10^6` phép (chấp nhận được). Bộ nhớ `O(n²)`. Dùng làm oracle vì rõ ràng theo đề.

**Nút thắt:** mỗi lần dịch ta xét cả những ô `0` (rất nhiều) và tạo ma trận mới. Ta chỉ quan tâm các ô `1`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bỏ phiếu cho vector dịch** (mục 4): với mỗi cặp ô `1` `(a, b)` tăng bộ đếm của vector `b − a`. Đáp án là bộ đếm lớn nhất. Số cặp tối đa `|A|·|B| ≤ n²·n² = 810 000` ở `n = 30` — nhưng thường nhỏ hơn rất nhiều nếu ảnh thưa. Không cần xét biên, không cần tạo ma trận.

Các hướng tăng tốc khác: dùng **bit** (mỗi hàng là một số nguyên, AND và `popcount` xử lý cả hàng một lúc) và **tích chập** (mọi độ chồng cùng lúc bằng một phép nhân số nguyên lớn).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** bỏ phiếu vector dịch (`Counter`), **Cách 3** bitmask theo hàng, **Cách 4** tương quan chéo bằng nhân số nguyên lớn.

### Cách 2 — Bỏ phiếu vector dịch

**Giải thích từng đoạn:**

1. Liệt kê toạ độ các ô `1` của hai ảnh: `A`, `B`.
2. Với mọi cặp `(a, b)`: `count[(b_i − a_i, b_j − a_j)] += 1`.
3. Đáp án `max(count.values(), default=0)` (`0` nếu một ảnh không có ô `1`).

In [9]:
def overlap_vote(img1, img2):
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    count = Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B)
    return max(count.values(), default=0)


print(overlap_vote(img1, img2), overlap_vote([[1]], [[1]]), overlap_vote([[0]], [[0]]))
assert overlap_vote(img1, img2) == 3 and overlap_vote([[1]], [[1]]) == 1 and overlap_vote([[0]], [[0]]) == 0

3 1 0


In [10]:
# Code nộp LeetCode cho Cách 2
from collections import Counter


class SolutionVote:
    def largestOverlap(self, img1: List[List[int]], img2: List[List[int]]) -> int:
        n = len(img1)
        a = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
        b = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
        cnt = Counter((x2 - x1, y2 - y1) for x1, y1 in a for x2, y2 in b)
        return max(cnt.values(), default=0)

**Độ phức tạp:** `O(|A|·|B|)` thời gian (tối đa `n⁴ = 810 000`), bộ nhớ `O((2n−1)²)` (số vector khác nhau). Gọn, dễ nhớ, và là cách chuẩn.

### Cách 3 — Bitmask theo hàng

**Ý tưởng:** mỗi hàng là một số nguyên `n` bit. Với phép dịch `(dx, dy)` (dịch ảnh 1 tới ảnh 2): hàng `i` của ảnh 1 trùng hàng `i + dx` của ảnh 2. Dịch cột `dy`: nếu `dy ≥ 0` thì `row1 << dy` (cắt bằng mặt nạ `n` bit), nếu `dy < 0` thì `row1 >> (−dy)`. Độ chồng của hai hàng là `(row1' & row2).bit_count()`. Cộng theo các hàng hợp lệ.

Ghi chú hướng bit: bit `j` ↔ cột `j`, nên dịch cột `+dy` là dịch **trái** `dy` bit (`<<`).

In [11]:
def overlap_bits(img1, img2):
    n = len(img1)
    mask = (1 << n) - 1
    r1 = [sum(v << j for j, v in enumerate(row)) for row in img1]
    r2 = [sum(v << j for j, v in enumerate(row)) for row in img2]
    best = 0
    for dx in range(-(n - 1), n):
        lo, hi = max(0, -dx), min(n, n - dx)               # các hàng i của ảnh 1 có i + dx trong [0, n)
        for dy in range(-(n - 1), n):
            total = 0
            for i in range(lo, hi):
                row = (r1[i] << dy) & mask if dy >= 0 else r1[i] >> (-dy)
                total += (row & r2[i + dx]).bit_count()
            if total > best:
                best = total
    return best


print(overlap_bits(img1, img2), overlap_bits([[1]], [[1]]), overlap_bits([[0]], [[0]]))
assert overlap_bits(img1, img2) == 3 and overlap_bits([[1]], [[1]]) == 1 and overlap_bits([[0]], [[0]]) == 0

3 1 0


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionBits:
    def largestOverlap(self, img1: List[List[int]], img2: List[List[int]]) -> int:
        n = len(img1)
        mask = (1 << n) - 1
        r1 = [sum(v << j for j, v in enumerate(row)) for row in img1]
        r2 = [sum(v << j for j, v in enumerate(row)) for row in img2]
        ans = 0
        for dx in range(-(n - 1), n):
            lo, hi = max(0, -dx), min(n, n - dx)
            for dy in range(-(n - 1), n):
                total = 0
                for i in range(lo, hi):
                    row = (r1[i] << dy) & mask if dy >= 0 else r1[i] >> (-dy)
                    total += bin(row & r2[i + dx]).count("1")
                ans = max(ans, total)
        return ans

**Độ phức tạp:** `(2n−1)²` phép dịch × `O(n)` hàng = `O(n³)` thao tác trên số nguyên `n` bit (`n ≤ 30` nên là một từ máy): `~10^5` phép ở `n = 30` — nhanh hơn Cách 1 cỡ `n` lần. Bộ nhớ `O(n)`.

### Cách 4 — Tương quan chéo bằng một phép nhân số nguyên lớn

**Ý tưởng khác:** độ chồng với mọi phép dịch là **tương quan chéo** (cross-correlation) của hai ảnh. Biểu diễn mỗi ảnh như đa thức hai biến, **đóng gói** thành một số nguyên lớn theo cơ số `2^k`, nhân hai số, rồi **đọc các hệ số**. Chi tiết:

1. Chiều rộng hàng `W = 2n − 1` (để không "tràn dòng" khi cộng toạ độ cột, vì cột tích nằm trong `[0, 2n−2]`).
2. Ảnh 1: ô `(i, j)` ↔ số mũ `i·W + j`. Ảnh 2 **lật hai chiều**: ô `(i', j')` ↔ số mũ `(n−1−i')·W + (n−1−j')`.
3. Hệ số của tích tại số mũ `P = (n−1−dx)·W + (n−1−dy)` là số cặp `(a ∈ A, b ∈ B)` với `b − a = (dx, dy)` = độ chồng của phép dịch `(dx, dy)`.
4. Mỗi hệ số `≤ n² = 900 < 2^10`; chọn `k = 11` để không có nhớ. Đáp án là hệ số lớn nhất.

In [13]:
def overlap_bigint(img1, img2):
    n = len(img1)
    W = 2 * n - 1
    k = 11                                                  # hệ số <= n^2 = 900 < 2^11: không nhớ
    x = y = 0
    for i in range(n):
        for j in range(n):
            if img1[i][j]:
                x |= 1 << (k * (i * W + j))
            if img2[i][j]:
                y |= 1 << (k * ((n - 1 - i) * W + (n - 1 - j)))
    prod = x * y
    mask = (1 << k) - 1
    best = 0
    total_positions = (2 * n - 1) * W
    for pos in range(total_positions):
        c = (prod >> (k * pos)) & mask
        if c > best:
            best = c
    return best


print(overlap_bigint(img1, img2), overlap_bigint([[1]], [[1]]), overlap_bigint([[0]], [[0]]))
assert overlap_bigint(img1, img2) == 3 and overlap_bigint([[1]], [[1]]) == 1 and overlap_bigint([[0]], [[0]]) == 0

3 1 0


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionBigInt:
    def largestOverlap(self, img1: List[List[int]], img2: List[List[int]]) -> int:
        n = len(img1)
        w, k = 2 * n - 1, 11
        x = y = 0
        for i in range(n):
            for j in range(n):
                if img1[i][j]:
                    x |= 1 << (k * (i * w + j))
                if img2[i][j]:
                    y |= 1 << (k * ((n - 1 - i) * w + (n - 1 - j)))
        prod, mask, ans = x * y, (1 << k) - 1, 0
        for pos in range((2 * n - 1) * w):
            ans = max(ans, (prod >> (k * pos)) & mask)
        return ans

**Độ phức tạp:** đóng gói `O(n²)`; một phép nhân hai số `~ k·(2n)² ≈ 40 000` bit (nhanh); đọc `(2n−1)·W ≈ 3500` hệ số. Về lý thuyết `O(n² log n)`-kiểu nhờ nhân nhanh, thực tế vài chục micro-giây ở `n = 30`. Đây là ý tưởng của **FFT/NTT** nhưng dùng số nguyên lớn có sẵn trong Python.

## 8. TRACE từng bước

**Trace Cách 2 (bỏ phiếu)** cho ví dụ 1 (4 ô `1` ở `img1`, 3 ô `1` ở `img2`, tổng 12 phiếu):

| ô `a` của ảnh 1 | phiếu cho các vector `b − a` với `b ∈ {(1,1), (1,2), (2,2)}` |
|---|---|
| `(0,0)` | `(1,1)`, `(1,2)`, `(2,2)` |
| `(0,1)` | `(1,0)`, `(1,1)`, `(2,1)` |
| `(1,1)` | `(0,0)`, `(0,1)`, `(1,1)` |
| `(2,1)` | `(-1,0)`, `(-1,1)`, `(0,1)` |

Số phiếu: `(1,1)` được `3`; `(0,1)` được `2`; các vector còn lại `1`. Max `= 3`. Hàm trace in từng bước tích luỹ:

In [15]:
def trace_vote(img1, img2):
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    print("A =", A, "| B =", B)
    count = Counter()
    for a in A:
        row = []
        for b in B:
            v = (b[0] - a[0], b[1] - a[1])
            count[v] += 1
            row.append(f"{v}->{count[v]}")
        print(f"ô a={a}: " + ", ".join(row))
    top = count.most_common(1)[0] if count else (None, 0)
    print("vector nhiều phiếu nhất:", top)
    return top[1]


assert trace_vote(img1, img2) == 3

A = [(0, 0), (0, 1), (1, 1), (2, 1)] | B = [(1, 1), (1, 2), (2, 2)]
ô a=(0, 0): (1, 1)->1, (1, 2)->1, (2, 2)->1
ô a=(0, 1): (1, 0)->1, (1, 1)->2, (2, 1)->1
ô a=(1, 1): (0, 0)->1, (0, 1)->1, (1, 1)->3
ô a=(2, 1): (-1, 0)->1, (-1, 1)->1, (0, 1)->2
vector nhiều phiếu nhất: ((1, 1), 3)


**Trace Cách 3 (bitmask)** cho phép dịch tốt nhất `(dx, dy) = (1, 1)` trên ví dụ 1: các hàng của `img1` (bit `j` ↔ cột `j`): `[0b011, 0b010, 0b010]`; của `img2`: `[0b000, 0b110, 0b100]`. Dịch cột `+1` (`<< 1`, giữ 3 bit): `0b110, 0b100, 0b100`. Hàng `i` của ảnh 1 ghép với hàng `i + 1` của ảnh 2:

| `i` | `row1 << 1` | hàng `i+1` của ảnh 2 | AND | số bit 1 |
|---|---|---|---|---|
| 0 | `110` | `110` | `110` | 2 |
| 1 | `100` | `100` | `100` | 1 |
| 2 | — (không có hàng 3) | — | — | 0 |

Tổng `3` ✓. Hàm trace in đúng các số này:

In [16]:
def trace_bits(img1, img2, dx, dy):
    n = len(img1)
    mask = (1 << n) - 1
    r1 = [sum(v << j for j, v in enumerate(row)) for row in img1]
    r2 = [sum(v << j for j, v in enumerate(row)) for row in img2]
    print("hàng ảnh 1:", [f"{r:0{n}b}" for r in r1], "| hàng ảnh 2:", [f"{r:0{n}b}" for r in r2])
    total = 0
    for i in range(n):
        if not (0 <= i + dx < n):
            print(f"i={i}: hàng {i + dx} ngoài ảnh -> bỏ")
            continue
        row = (r1[i] << dy) & mask if dy >= 0 else r1[i] >> (-dy)
        both = row & r2[i + dx]
        total += both.bit_count()
        print(f"i={i}: dịch cột -> {row:0{n}b}, hàng {i + dx} của ảnh 2 = {r2[i + dx]:0{n}b}, AND = {both:0{n}b} ({both.bit_count()} bit 1)")
    print("tổng độ chồng của phép dịch", (dx, dy), "=", total)
    return total


assert trace_bits(img1, img2, 1, 1) == 3

hàng ảnh 1: ['011', '010', '010'] | hàng ảnh 2: ['000', '110', '100']
i=0: dịch cột -> 110, hàng 1 của ảnh 2 = 110, AND = 110 (2 bit 1)
i=1: dịch cột -> 100, hàng 2 của ảnh 2 = 100, AND = 100 (1 bit 1)
i=2: hàng 3 ngoài ảnh -> bỏ
tổng độ chồng của phép dịch (1, 1) = 3


**Trace Cách 4 (nhân số nguyên lớn)**: ta in các hệ số khác `0` của tích, kèm phép dịch `(dx, dy)` tương ứng với vị trí `P`:

In [17]:
def trace_bigint(img1, img2):
    n = len(img1)
    W, k = 2 * n - 1, 11
    x = y = 0
    for i in range(n):
        for j in range(n):
            if img1[i][j]:
                x |= 1 << (k * (i * W + j))
            if img2[i][j]:
                y |= 1 << (k * ((n - 1 - i) * W + (n - 1 - j)))
    prod = x * y
    mask = (1 << k) - 1
    found = {}
    for pos in range((2 * n - 1) * W):
        c = (prod >> (k * pos)) & mask
        if c:
            row, col = divmod(pos, W)
            dx, dy = n - 1 - row, n - 1 - col
            found[(dx, dy)] = c
    print("các hệ số khác 0 (phép dịch -> độ chồng):", dict(sorted(found.items())))
    return max(found.values(), default=0)


assert trace_bigint(img1, img2) == 3

các hệ số khác 0 (phép dịch -> độ chồng): {(-1, 0): 1, (-1, 1): 1, (0, 0): 1, (0, 1): 2, (1, 0): 1, (1, 1): 3, (1, 2): 1, (2, 1): 1, (2, 2): 1}


## 9. Code nộp LeetCode

Chọn **bỏ phiếu vector dịch** (ngắn gọn, `O(n⁴)` tệ nhất nhưng thực tế rất nhanh). Giữ nguyên `class Solution`, `largestOverlap(self, img1, img2)`:

In [18]:
from collections import Counter
from typing import List


class Solution:
    def largestOverlap(self, img1: List[List[int]], img2: List[List[int]]) -> int:
        n = len(img1)
        a = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]    # toạ độ ô 1 của ảnh 1
        b = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]    # toạ độ ô 1 của ảnh 2
        cnt = Counter((x2 - x1, y2 - y1) for x1, y1 in a for x2, y2 in b)  # mỗi cặp bỏ phiếu cho vector dịch
        return max(cnt.values(), default=0)


print(Solution().largestOverlap(img1, img2), Solution().largestOverlap([[1]], [[1]]), Solution().largestOverlap([[0]], [[0]]))
assert Solution().largestOverlap(img1, img2) == 3
assert Solution().largestOverlap([[1]], [[1]]) == 1 and Solution().largestOverlap([[0]], [[0]]) == 0

3 1 0


## 10. Vì sao đúng

**Định lý.** Với phép dịch `v = (dx, dy)` (ảnh 1 dịch đi `v`), độ chồng bằng số cặp `(a, b) ∈ A × B` với `b − a = v`.

*Chứng minh.* Sau khi dịch, ô `1` của ảnh 1 nằm ở `a + v` (nếu còn trong ma trận). Vị trí này chồng với ô `1` của ảnh 2 nếu và chỉ nếu `a + v ∈ B`, tức là tồn tại `b ∈ B` với `b = a + v`. Khi `a + v ∈ B` thì `a + v` là toạ độ trong ma trận nên ô không bị xoá; ngược lại, ô bị xoá không thể ∈ B. Vì mỗi `a` có đúng một `a + v` nên đếm theo `a` hay theo cặp `(a, b)` với `b − a = v` là như nhau (mỗi `a` đóng góp tối đa 1, và đúng 1 nếu `a + v ∈ B`). ∎

Hệ quả: đáp án `max_v #{(a, b): b − a = v}` = tần suất lớn nhất của vector trong đa tập `{b − a}`; nếu `A` hoặc `B` rỗng thì đáp án `0`.

**Phạm vi dịch.** Vector `b − a` có mỗi toạ độ trong `[−(n−1), n−1]`, đúng phạm vi mà Cách 1 và 3 duyệt. Dịch xa hơn làm ảnh ra hết ngoài (độ chồng `0`).

**Cách 3.** Với `(dx, dy)`: hàng `i` ↔ hàng `i + dx` (bỏ nếu ngoài); `<< dy` dịch cột sang phải `dy` ô (bit `j` → `j + dy`), mặt nạ `n` bit xoá phần tràn; `>>` dịch trái; `AND` + `bit_count` đếm ô `1` chung của hàng.

**Cách 4.** Hệ số của `x·y` ở số mũ `P` bằng `Σ [a ở số mũ s]·[b' ở số mũ t]` với `s + t = P`. Ảnh 2 lật: `t = (n−1−i')·W + (n−1−j')`. Với `a = (i, j)`: `s + t = (i + n − 1 − i')·W + (j + n − 1 − j') = (n−1−dx)·W + (n−1−dy)` với `(dx, dy) = (i' − i, j' − j)`. Vì `(n−1−dy) ∈ [0, 2n−2] < W` nên `(dx, dy)` được xác định duy nhất bởi `P` (không đụng nhau giữa các hàng). Mỗi hệ số ≤ `n² < 2^11` nên không có nhớ giữa các ô.

Ta kiểm chứng bằng máy: 4 cách khớp **oracle dịch ảnh thật** trên mọi cặp ảnh `2 × 2`, 5000 cặp `3 × 3` và hàng nghìn ảnh ngẫu nhiên tới `n = 30` (mục 12).

## 11. Time / Space Complexity

Đặt `n` là cạnh ảnh, `p = |A|`, `q = |B|` số ô `1` (`p, q ≤ n²`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi phép dịch | `O(n⁴)` (`(2n−1)² · n²`) | `O(n²)` | oracle |
| 2. Bỏ phiếu vector | `O(p·q)` ≤ `O(n⁴)` | `O(min(p·q, n²))` | **đáp án chính**; nhanh nếu ảnh thưa |
| 3. Bitmask theo hàng | `O(n³)` | `O(n)` | nhanh nhất khi ảnh đặc |
| 4. Nhân số nguyên lớn | `O(M(n² k))` + đọc `O(n²)` | `O(n² k)` bit | tương quan chéo, mở rộng cho ảnh lớn |

Đo thực tế ở `n = 30` với ảnh thưa, vừa, đặc:

In [19]:
def random_img(rng, n, p):
    return [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(n)]


def bench(fn, a, b, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(a, b)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(835)
for p in (0.1, 0.5, 0.9):
    a, b = random_img(rng, 30, p), random_img(rng, 30, p)
    ref = overlap_vote(a, b)
    print(f"n=30, mật độ {p}: đáp án = {ref}")
    for name, fn in [("brute", overlap_brute), ("vote", overlap_vote), ("bits", overlap_bits), ("bigint", overlap_bigint)]:
        assert fn(a, b) == ref
        print(f"   {name:7s} {bench(fn, a, b, 1 if name == 'brute' else 3) * 1000:8.2f} ms")

n=30, mật độ 0.1: đáp án = 13


   brute     295.43 ms
   vote        0.82 ms
   bits        5.41 ms
   bigint      2.29 ms
n=30, mật độ 0.5: đáp án = 240


   brute     372.30 ms
   vote       23.48 ms
   bits        5.33 ms
   bigint      2.50 ms
n=30, mật độ 0.9: đáp án = 727


   brute     456.67 ms


   vote       79.68 ms
   bits        9.41 ms
   bigint      4.31 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) vét cạn mọi cặp ảnh nhỏ, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn lớn nhất. **Oracle:** `overlap_brute` (dịch ảnh thật sự, xoá phần tràn).

### A. Ví dụ của đề

In [20]:
IMPLS = {
    "brute": overlap_brute,
    "vote": overlap_vote,
    "bits": overlap_bits,
    "bigint": overlap_bigint,
    "Solution": lambda a, b: Solution().largestOverlap(a, b),
    "SolutionVote": lambda a, b: SolutionVote().largestOverlap(a, b),
    "SolutionBits": lambda a, b: SolutionBits().largestOverlap(a, b),
    "SolutionBigInt": lambda a, b: SolutionBigInt().largestOverlap(a, b),
}


def check(a, b, want=None, names=None):
    if want is None:
        want = overlap_brute(a, b)
    for name in (names or IMPLS):
        got = IMPLS[name](a, b)
        assert got == want, (name, a, b, got, want)


check(img1, img2, 3)
check([[1]], [[1]], 1)
check([[0]], [[0]], 0)
print("ví dụ 1, 2, 3: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2, 3: cả 8 cài đặt đúng


### B. Edge case

- `n = 1`: `[[1]]` với `[[1]]` → `1`; `[[1]]` với `[[0]]` → `0`.
- Một trong hai ảnh toàn `0` → `0`.
- Hai ảnh toàn `1` → `n²` (không dịch).
- Ảnh có một ô `1` ở các góc đối diện (cần dịch tối đa `n−1` mỗi chiều).
- Ảnh đối xứng: cùng ảnh thì độ chồng `= số ô 1` (dịch `0`).
- Hai ảnh giống nhau nhưng lệch (ảnh 2 = ảnh 1 dịch).

In [21]:
n = 6
zeros = [[0] * n for _ in range(n)]
ones = [[1] * n for _ in range(n)]
corner_tl = [[1 if (i, j) == (0, 0) else 0 for j in range(n)] for i in range(n)]
corner_br = [[1 if (i, j) == (n - 1, n - 1) else 0 for j in range(n)] for i in range(n)]
rnd = random_img(random.Random(3), n, 0.5)
cases = [
    ([[1]], [[0]], 0), ([[0]], [[1]], 0),
    (zeros, rnd, 0), (rnd, zeros, 0),
    (ones, ones, n * n),
    (corner_tl, corner_br, 1),                      # dịch (n-1, n-1) để chồng
    (rnd, rnd, sum(map(sum, rnd))),                 # cùng ảnh: không dịch
    (rnd, shifted(rnd, 1, 2), sum(map(sum, shifted(rnd, 1, 2)))),   # ảnh 2 = ảnh 1 dịch (phần còn lại chồng hết)
]
for a, b, want in cases:
    check(a, b, want)
print(len(cases), "edge case: tất cả cài đặt đúng")

8 edge case: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [22]:
def bug_positive_only(img1, img2):                  # chỉ thử dịch xuống/phải (dx, dy >= 0)
    n = len(img1)
    return max(overlap(shifted(img1, dx, dy), img2) for dx in range(0, n) for dy in range(0, n))


def bug_cyclic(img1, img2):                         # dịch vòng (wrap-around) thay vì xoá
    n = len(img1)
    best = 0
    for dx in range(n):
        for dy in range(n):
            cnt = sum(img1[(i - dx) % n][(j - dy) % n] & img2[i][j] for i in range(n) for j in range(n))
            best = max(best, cnt)
    return best


def bug_count_zeros(img1, img2):                    # đếm cả cặp 0-0 là "khớp"
    n = len(img1)
    best = 0
    for dx in range(-(n - 1), n):
        for dy in range(-(n - 1), n):
            s = shifted(img1, dx, dy)
            best = max(best, sum(1 for i in range(n) for j in range(n) if s[i][j] == img2[i][j]))
    return best


def variant_negated_vote(img1, img2):                 # bỏ phiếu theo a - b thay vì b - a: KHÔNG phải lỗi (minh hoạ)
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    cnt = Counter((ai - bi, aj - bj) for ai, aj in A for bi, bj in B)
    cnt2 = Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B)
    assert sorted(cnt.values()) == sorted(cnt2.values())              # cùng đa tập số phiếu (chỉ đổi dấu khoá)
    return max(cnt.values(), default=0)


def bug_one_axis(img1, img2):                       # chỉ dịch theo hàng (dy = 0)
    n = len(img1)
    return max(overlap(shifted(img1, dx, 0), img2) for dx in range(-(n - 1), n))


BUGS = [bug_positive_only, bug_cyclic, bug_count_zeros, bug_one_axis]
rng = random.Random(1)
tests = [(random_img(rng, rng.randint(2, 5), 0.4), None) for _ in range(500)]
tests = [(a, random_img(rng, len(a), 0.4)) for a, _ in tests]
for bug in BUGS:
    bad = next(((a, b) for a, b in tests if bug(a, b) != overlap_brute(a, b)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên 1 ca: ra {bug(*bad)}, đúng {overlap_brute(*bad)}")
# bản "đổi dấu" thực ra đúng (vector đối xứng): không phải lỗi
assert all(variant_negated_vote(a, b) == overlap_brute(a, b) for a, b in tests)
print("variant_negated_vote: đảo dấu vector (a-b thay b-a) cho cùng đáp án — vì tần suất lớn nhất của v bằng của -v khi đổi vai trò ảnh")

bug_positive_only    sai trên 1 ca: ra 5, đúng 6
bug_cyclic           sai trên 1 ca: ra 9, đúng 6
bug_count_zeros      sai trên 1 ca: ra 7, đúng 4
bug_one_axis         sai trên 1 ca: ra 1, đúng 2
variant_negated_vote: đảo dấu vector (a-b thay b-a) cho cùng đáp án — vì tần suất lớn nhất của v bằng của -v khi đổi vai trò ảnh


### D. Vét cạn mọi cặp ảnh nhỏ

- **Mọi** cặp ảnh `2 × 2`: `16 × 16 = 256` cặp.
- **5000** cặp ảnh `3 × 3` lấy mẫu có seed trong tổng `512 × 512 = 262144`.
- Mọi cặp ảnh `1 × 1` (4 cặp).

In [23]:
cnt = 0
for n, total in [(1, None), (2, None)]:
    for a_bits in product((0, 1), repeat=n * n):
        for b_bits in product((0, 1), repeat=n * n):
            a = [list(a_bits[i * n:(i + 1) * n]) for i in range(n)]
            b = [list(b_bits[i * n:(i + 1) * n]) for i in range(n)]
            check(a, b)
            cnt += 1
print("mọi cặp 1x1 và 2x2:", cnt, "cặp khớp")

rng = random.Random(33)
for _ in range(5000):
    a, b = random_img(rng, 3, rng.choice([0.2, 0.5, 0.8])), random_img(rng, 3, rng.choice([0.2, 0.5, 0.8]))
    check(a, b)
print("5000 cặp 3x3 ngẫu nhiên khớp oracle dịch ảnh thật")

mọi cặp 1x1 và 2x2: 260 cặp khớp


5000 cặp 3x3 ngẫu nhiên khớp oracle dịch ảnh thật


### E. Ngẫu nhiên có seed, đúng ràng buộc và tính chất

In [24]:
rng = random.Random(20240835)
count = 0
for _ in range(500):
    n = rng.randint(1, 30)
    a, b = random_img(rng, n, rng.choice([0.05, 0.2, 0.5, 0.9])), random_img(rng, n, rng.choice([0.05, 0.2, 0.5, 0.9]))
    want = overlap_vote(a, b)
    for name in ("vote", "bits", "bigint", "Solution", "SolutionVote", "SolutionBits", "SolutionBigInt"):
        assert IMPLS[name](a, b) == want, (name, n)
    if n <= 14:
        assert overlap_brute(a, b) == want
    count += 1
print(count, "cặp ảnh ngẫu nhiên (n tới 30): mọi cài đặt trùng nhau; brute đối chiếu khi n <= 14")

# tính chất: (1) đối xứng khi đổi vai trò ảnh; (2) <= min(số ô 1); (3) bằng khi ảnh 2 là ảnh 1; (4) bất biến khi dịch ảnh 2 (nếu không mất bit) ; (5) thêm bit không giảm
for _ in range(300):
    n = rng.randint(2, 10)
    a, b = random_img(rng, n, 0.4), random_img(rng, n, 0.4)
    ov = Solution().largestOverlap(a, b)
    assert ov == Solution().largestOverlap(b, a)
    assert ov <= min(sum(map(sum, a)), sum(map(sum, b)))
    assert Solution().largestOverlap(a, a) == sum(map(sum, a))
    a2 = [row[:] for row in a]
    zeros = [(i, j) for i in range(n) for j in range(n) if a2[i][j] == 0]
    if zeros:
        i, j = rng.choice(zeros)
        a2[i][j] = 1
        assert Solution().largestOverlap(a2, b) >= ov
print("tính chất: đối xứng, ≤ min số ô 1, tự chồng = số ô 1, thêm bit không giảm — OK")

500 cặp ảnh ngẫu nhiên (n tới 30): mọi cài đặt trùng nhau; brute đối chiếu khi n <= 14
tính chất: đối xứng, ≤ min số ô 1, tự chồng = số ô 1, thêm bit không giảm — OK


### F. Giới hạn lớn nhất

`n = 30`: ảnh toàn `1` (đáp án `900`), ảnh ngẫu nhiên đặc, ảnh có vài ô, ảnh có mẫu lặp (kiểm tra số phiếu lớn).

In [25]:
n = 30
rng = random.Random(777)
pattern = [[(i * 7 + j * 3) % 5 == 0 for j in range(n)] for i in range(n)]
pattern = [[int(v) for v in row] for row in pattern]
cases = {
    "toàn 1": ([[1] * n for _ in range(n)], [[1] * n for _ in range(n)]),
    "ngẫu nhiên đặc": (random_img(rng, n, 0.9), random_img(rng, n, 0.9)),
    "ngẫu nhiên thưa": (random_img(rng, n, 0.05), random_img(rng, n, 0.05)),
    "mẫu tuần hoàn": (pattern, [row[:] for row in pattern]),
    "mẫu dịch (1,2)": (pattern, shifted(pattern, 1, 2)),
}
for name, (a, b) in cases.items():
    want = overlap_brute(a, b)
    t0 = time.perf_counter()
    for impl in IMPLS:
        assert IMPLS[impl](a, b) == want, (name, impl)
    print(f"{name:18s}: đáp án {want:4d} (8 cài đặt khớp brute)")
assert overlap_brute(*cases["toàn 1"]) == 900

toàn 1            : đáp án  900 (8 cài đặt khớp brute)


ngẫu nhiên đặc    : đáp án  736 (8 cài đặt khớp brute)


ngẫu nhiên thưa   : đáp án    8 (8 cài đặt khớp brute)


mẫu tuần hoàn     : đáp án  180 (8 cài đặt khớp brute)


mẫu dịch (1,2)    : đáp án  163 (8 cài đặt khớp brute)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ thử dịch xuống/phải

Dịch ảnh 1 xuống/phải **không** đủ: có khi phải dịch **lên/trái** (hoặc ngược lại dịch ảnh 2). Phải thử cả hai dấu của `dx, dy`.

In [26]:
a = [[0, 0], [0, 1]]
b = [[1, 0], [0, 0]]
print("chỉ dịch dương:", bug_positive_only(a, b), "| đúng:", Solution().largestOverlap(a, b))
assert bug_positive_only(a, b) == 0 and Solution().largestOverlap(a, b) == 1

chỉ dịch dương: 0 | đúng: 1


**Vì sao sai:** ô `1` của ảnh 1 ở `(1,1)` cần dịch `(−1, −1)` để chồng lên `(0,0)` của ảnh 2. **Đúng:** `dx, dy ∈ [−(n−1), n−1]`, hoặc bỏ phiếu theo vector `b − a`.

### Lỗi 2: dịch vòng (wrap-around)

Dùng chỉ số modulo `(i − dx) % n` biến phép dịch thành **xoay vòng** (như `np.roll`). Đề nói bit trượt ra ngoài bị **xoá**, không quay lại đầu bên kia.

In [27]:
a = [[1, 0, 1], [0, 0, 0], [0, 0, 0]]      # hai ô cách nhau 2 cột = liền kề theo vòng (cột 2 và cột 0)
b = [[1, 1, 0], [0, 0, 0], [0, 0, 0]]      # hai ô liền kề thật
print("dịch vòng:", bug_cyclic(a, b), "| đúng:", Solution().largestOverlap(a, b))
assert bug_cyclic(a, b) == 2 and Solution().largestOverlap(a, b) == 1
found = next((x, y) for x, y in tests if bug_cyclic(x, y) != overlap_brute(x, y))
print("một ca ngẫu nhiên:", bug_cyclic(*found), "vs", overlap_brute(*found))

dịch vòng: 2 | đúng: 1
một ca ngẫu nhiên: 9 vs 6


**Vì sao sai:** với dịch vòng, bit trượt ra mép phải quay lại mép trái và có thể chồng lên bit khác, cho độ chồng **lớn hơn** thật. **Đúng:** nếu `(i + dx, j + dy)` ngoài ma trận thì bỏ.

### Lỗi 3: đếm cả cặp `0 – 0`

"Độ chồng" chỉ tính vị trí cả hai ảnh đều là `1`. Đếm ô giống nhau (kể cả `0=0`) cho kết quả phình to.

In [28]:
a, b = [[1, 0], [0, 0]], [[0, 1], [0, 0]]
print("đếm cả 0-0:", bug_count_zeros(a, b), "| đúng:", Solution().largestOverlap(a, b))
assert bug_count_zeros(a, b) > Solution().largestOverlap(a, b)

đếm cả 0-0: 4 | đúng: 1


**Vì sao sai:** ô `0` trùng ô `0` không phải "chồng bit". **Đúng:** `a & b` (cả hai là `1`).

### Lỗi 4: chỉ dịch theo một trục

Chỉ thử `dy = 0` (hoặc `dx = 0`) bỏ sót các phép dịch chéo.

In [29]:
a, b = [[1, 0], [0, 0]], [[0, 0], [0, 1]]
print("chỉ theo hàng:", bug_one_axis(a, b), "| đúng:", Solution().largestOverlap(a, b))
assert bug_one_axis(a, b) == 0 and Solution().largestOverlap(a, b) == 1

chỉ theo hàng: 0 | đúng: 1


**Vì sao sai:** muốn chồng `(0,0)` lên `(1,1)` cần dịch chéo. **Đúng:** thử mọi cặp `(dx, dy)`.

### Lỗi 5 (bonus): nhầm hướng của vector dịch

Vector dịch có thể tính là `b − a` (dịch ảnh 1 để chồng lên ảnh 2) hoặc `a − b` (dịch ảnh 2). Hai cách **đối xứng** cho cùng tần suất lớn nhất, nên bài này không bị lỗi vì nhầm dấu **nếu** dùng nhất quán một hướng cho mọi cặp. Lỗi thật xảy ra khi **trộn** hai hướng trong cùng một `Counter`.

In [30]:
def bug_mixed_direction(img1, img2):
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    cnt = Counter()
    for k, (ai, aj) in enumerate(A):
        for bi, bj in B:
            cnt[(bi - ai, bj - aj) if k % 2 == 0 else (ai - bi, aj - bj)] += 1        # trộn hướng
    return max(cnt.values(), default=0)


bad = next(((x, y) for x, y in tests if bug_mixed_direction(x, y) != overlap_brute(x, y)), None)
print("trộn hướng sai trên 1 ca:", bug_mixed_direction(*bad), "vs đúng", overlap_brute(*bad))

trộn hướng sai trên 1 ca: 2 vs đúng 3


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tìm phép biến đổi (tịnh tiến) làm hai tập điểm khớp nhiều nhất" → **bỏ phiếu cho vector biến đổi** (Hough transform): mỗi cặp điểm bỏ phiếu cho phép dịch đưa điểm này tới điểm kia; phép dịch nhiều phiếu nhất là đáp án.

**Mẹo:**

1. Chuyển bài ma trận sang **tập toạ độ ô `1`**; ma trận thưa thì rất nhanh.
2. Vector dịch `b − a` tự động tôn trọng biên (không cần kiểm tra) vì chỉ tính các ô thật sự chồng.
3. Ảnh đặc: dùng **bitmask** theo hàng (`AND` + `bit_count`).
4. Nhiều phép dịch cùng lúc = **tích chập/tương quan chéo** → FFT hoặc nhân số nguyên lớn.
5. Không có phép xoay; nếu có, thử 4 góc xoay.
6. Oracle tốt: **dịch ảnh thật** (xoá phần tràn) rồi đếm AND.

**Họ bài cùng pattern:** Rotate Image (48), Flipping an Image (832), Number of Distinct Islands (694, so khớp hình bằng dịch), Count Servers that Communicate (1267), Smallest Rectangle Enclosing Black Pixels (302).

## 15. Biến thể

1. **Cho phép xoay** (bội của `90°`) rồi dịch: thử `4` góc xoay bằng cách xoay ảnh 1 rồi bỏ phiếu.
2. **Trả về chính phép dịch** tối ưu `(dx, dy)` (và độ chồng).
3. **Ảnh chữ nhật** `m × n`: dịch trong `[−(m−1), m−1] × [−(n−1), n−1]`.
4. **Cho phép dịch cả hai ảnh** — vẫn như một ảnh (dịch tương đối).
5. **Số phép dịch đạt tối ưu** (đếm số vector đạt max).

Cài đặt biến thể 1 và 2:

In [31]:
def rotate90(a):
    return [list(row) for row in zip(*a[::-1])]


def largest_overlap_with_rotation(img1, img2):
    best = 0
    cur = img1
    for _ in range(4):
        best = max(best, overlap_vote(cur, img2))
        cur = rotate90(cur)
    return best


def best_shift(img1, img2):
    """Biến thể 2: trả (độ chồng, một phép dịch (dx, dy) đạt tối ưu); (0, None) nếu không có phiếu."""
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    cnt = Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B)
    if not cnt:
        return 0, None
    v, c = max(cnt.items(), key=lambda kv: (kv[1], kv[0]))
    return c, v


print(best_shift(img1, img2))
c, v = best_shift(img1, img2)
assert overlap(shifted(img1, *v), img2) == c == 3
img_l = [[1, 1, 0], [0, 0, 0], [0, 0, 0]]
img_v = [[1, 0, 0], [1, 0, 0], [0, 0, 0]]
print("không xoay:", overlap_vote(img_l, img_v), "| cho phép xoay:", largest_overlap_with_rotation(img_l, img_v))
assert overlap_vote(img_l, img_v) == 1 and largest_overlap_with_rotation(img_l, img_v) == 2

(3, (1, 1))
không xoay: 1 | cho phép xoay: 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi phép dịch | Dịch ảnh thật, đếm AND | `O(n⁴)` | `O(n²)` | Oracle |
| 2. Bỏ phiếu vector | `Counter` của `b − a` | `O(p·q)` ≤ `O(n⁴)` | `O(n²)` | **Đáp án chính**; ảnh thưa rất nhanh |
| 3. Bitmask theo hàng | `AND` + `bit_count` cả hàng | `O(n³)` | `O(n)` | Ảnh đặc, `n ≤ 64` |
| 4. Nhân số nguyên lớn | Tương quan chéo bằng một phép nhân | `O(M(n²·k))` | `O(n²·k)` bit | Mở rộng cho ảnh lớn / nhiều phép dịch |

Cả bốn khớp nhau và khớp oracle dịch ảnh thật trên mọi cặp `2 × 2`, 5000 cặp `3 × 3` và 500 cặp tới `n = 30` (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [32]:
a = [[1, 0], [1, 1]]
b = [[0, 1], [0, 1]]
A = [(i, j) for i in range(2) for j in range(2) if a[i][j]]
B = [(i, j) for i in range(2) for j in range(2) if b[i][j]]
print(A, B)
print(sorted(Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B).items()))
print(Solution().largestOverlap(a, b))

[(0, 0), (1, 0), (1, 1)] [(0, 1), (1, 1)]
[((-1, 0), 1), ((-1, 1), 1), ((0, 0), 1), ((0, 1), 2), ((1, 1), 1)]
2


### Bài 2 (tính tay)

`img1 = [[1,1,0],[1,0,0],[0,0,0]]`, `img2 = [[0,0,0],[0,1,1],[0,1,0]]` (hình chữ L cùng chiều).

- (a) Liệt kê `A` và `B`.
- (b) Có bao nhiêu phiếu mỗi vector? Vector nào nhiều nhất?
- (c) Đáp án và phép dịch tương ứng.

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện bỏ phiếu:

In [33]:
def vote_todo(img1, img2):
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    count = Counter()
    for ai, aj in A:
        for bi, bj in B:
            pass            # TODO 1: tăng phiếu của vector (bi - ai, bj - aj)
    return None             # TODO 2: trả tần suất lớn nhất (0 nếu không có phiếu)


# Khi làm xong, bỏ comment để kiểm tra:
# assert vote_todo(img1, img2) == 3 and vote_todo([[0]], [[0]]) == 0

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng bitmask theo hàng (hoặc bỏ phiếu):

In [34]:
class MySolution:
    def largestOverlap(self, img1, img2):
        pass            # TODO


# Khi làm xong:
# assert MySolution().largestOverlap(img1, img2) == 3
# assert MySolution().largestOverlap([[1]], [[1]]) == 1

### Bài 5 (mini challenge): đếm số phép dịch tối ưu

Viết `count_best_shifts(img1, img2)` trả về `(độ chồng lớn nhất, số phép dịch (dx, dy) đạt độ chồng đó)`. Chú ý: nếu độ chồng lớn nhất là `0` thì mọi phép dịch trong `[−(n−1), n−1]²` (tổng `(2n−1)²` phép) đều đạt; xử lý trường hợp này riêng. Kiểm bằng cách thử mọi phép dịch.

In [35]:
def count_best_shifts(img1, img2):
    pass            # TODO


# Khi làm xong:
# assert count_best_shifts([[0]], [[0]]) == (0, 1)
# assert count_best_shifts(img1, img2) == (3, 1)

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: liệt kê ô `1` rồi mọi hiệu `b − a`.
- Mức 2: `A = [(0,0),(1,0),(1,1)]`, `B = [(0,1),(1,1)]`.
- Mức 3: các vector: từ `(0,0)`: `(0,1), (1,1)`; từ `(1,0)`: `(-1,1), (0,1)`; từ `(1,1)`: `(-1,0), (0,0)`. Vector `(0,1)` xuất hiện `2` lần → đáp án `2`.

**Bài 2**

- Mức 1: `A` có 3 ô, `B` có 3 ô: 9 phiếu.
- Mức 2: nhận ra `B` chính là `A` dịch đi `(+1, +1)`.
- Mức 3: vector `(1,1)` nhận `3` phiếu (mỗi ô của `A` tới ô tương ứng của `B`).

**Bài 3**

- Mức 1: `count[(bi - ai, bj - aj)] += 1`.
- Mức 2: kết thúc `max(count.values(), default=0)`.
- Mức 3: không cần kiểm tra biên.

**Bài 4**

- Mức 1: đổi mỗi hàng thành số nguyên.
- Mức 2: `dy ≥ 0` thì `<<`, ngược lại `>>`; nhớ mặt nạ `n` bit.
- Mức 3: `total += (row & r2[i + dx]).bit_count()`.

**Bài 5**

- Mức 1: dùng `Counter` các vector.
- Mức 2: số vector đạt max là `sum(1 for c in cnt.values() if c == best)` khi `best > 0`.
- Mức 3: khi `best == 0` đáp án là `(0, (2n − 1)²)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [36]:
a = [[1, 0], [1, 1]]
b = [[0, 1], [0, 1]]
A = [(i, j) for i in range(2) for j in range(2) if a[i][j]]
B = [(i, j) for i in range(2) for j in range(2) if b[i][j]]
assert A == [(0, 0), (1, 0), (1, 1)] and B == [(0, 1), (1, 1)]
votes = Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B)
assert votes[(1, 1)] == 1 and max(votes.values()) == Solution().largestOverlap(a, b) == overlap_brute(a, b)
print("Bài 1:", sorted(votes.items()), "-> đáp án", max(votes.values()))

Bài 1: [((-1, 0), 1), ((-1, 1), 1), ((0, 0), 1), ((0, 1), 2), ((1, 1), 1)] -> đáp án 2


### Bài 2

(a) `A = {(0,0),(0,1),(1,0)}`, `B = {(1,1),(1,2),(2,1)}`. (b) 9 phiếu; vector `(1,1)` được `3` phiếu (các cặp `(0,0)→(1,1)`, `(0,1)→(1,2)`, `(1,0)→(2,1)`), các vector khác ít hơn. (c) Đáp án `3` với phép dịch `(+1, +1)`.

In [37]:
i1 = [[1, 1, 0], [1, 0, 0], [0, 0, 0]]
i2 = [[0, 0, 0], [0, 1, 1], [0, 1, 0]]
A = [(i, j) for i in range(3) for j in range(3) if i1[i][j]]
B = [(i, j) for i in range(3) for j in range(3) if i2[i][j]]
assert A == [(0, 0), (0, 1), (1, 0)] and B == [(1, 1), (1, 2), (2, 1)]
votes = Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B)
assert sum(votes.values()) == 9 and votes[(1, 1)] == 3 and max(votes.values()) == 3
assert Solution().largestOverlap(i1, i2) == overlap_brute(i1, i2) == 3
assert overlap(shifted(i1, 1, 1), i2) == 3
print("Bài 2 OK: vector (1,1) có", votes[(1, 1)], "phiếu; mọi vector:", sorted(votes.items()))

Bài 2 OK: vector (1,1) có 3 phiếu; mọi vector: [((0, 1), 1), ((0, 2), 1), ((1, 0), 1), ((1, 1), 3), ((1, 2), 1), ((2, 0), 1), ((2, 1), 1)]


### Bài 3

In [38]:
def vote_answer(img1, img2):
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    count = Counter()
    for ai, aj in A:
        for bi, bj in B:
            count[(bi - ai, bj - aj)] += 1
    return max(count.values(), default=0)


assert vote_answer(img1, img2) == 3 and vote_answer([[0]], [[0]]) == 0
rng = random.Random(3)
for _ in range(300):
    a, b = random_img(rng, 4, 0.4), random_img(rng, 4, 0.4)
    assert vote_answer(a, b) == overlap_brute(a, b)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def largestOverlap(self, img1, img2):
        n = len(img1)
        mask = (1 << n) - 1
        r1 = [sum(v << j for j, v in enumerate(row)) for row in img1]
        r2 = [sum(v << j for j, v in enumerate(row)) for row in img2]
        best = 0
        for dx in range(-(n - 1), n):
            for dy in range(-(n - 1), n):
                total = 0
                for i in range(n):
                    if 0 <= i + dx < n:
                        row = (r1[i] << dy) & mask if dy >= 0 else r1[i] >> (-dy)
                        total += (row & r2[i + dx]).bit_count()
                best = max(best, total)
        return best


assert MySolutionAnswer().largestOverlap(img1, img2) == 3 and MySolutionAnswer().largestOverlap([[1]], [[1]]) == 1
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 8)
    a, b = random_img(rng, n, 0.4), random_img(rng, n, 0.4)
    assert MySolutionAnswer().largestOverlap(a, b) == overlap_brute(a, b)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Nếu `best > 0` đếm các vector có đúng `best` phiếu (vector khác không có phiếu cho độ chồng `< best`). Nếu `best == 0`, mọi phép dịch (tổng `(2n−1)²`) đều cho `0`.

In [40]:
def count_best_shifts_answer(img1, img2):
    n = len(img1)
    A = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
    B = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
    cnt = Counter((bi - ai, bj - aj) for ai, aj in A for bi, bj in B)
    best = max(cnt.values(), default=0)
    if best == 0:
        return 0, (2 * n - 1) ** 2
    return best, sum(1 for c in cnt.values() if c == best)


def count_best_oracle(img1, img2):
    n = len(img1)
    vals = [overlap(shifted(img1, dx, dy), img2) for dx in range(-(n - 1), n) for dy in range(-(n - 1), n)]
    best = max(vals)
    return best, vals.count(best)


assert count_best_shifts_answer([[0]], [[0]]) == (0, 1)
assert count_best_shifts_answer(img1, img2) == (3, 1)
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 6)
    a, b = random_img(rng, n, rng.choice([0.0, 0.3, 0.7])), random_img(rng, n, rng.choice([0.0, 0.3, 0.7]))
    assert count_best_shifts_answer(a, b) == count_best_oracle(a, b), (a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (bỏ phiếu vector dịch):**

```python
class Solution:
    def largestOverlap(self, img1, img2):
        n = len(img1)
        a = [(i, j) for i in range(n) for j in range(n) if img1[i][j]]
        b = [(i, j) for i in range(n) for j in range(n) if img2[i][j]]
        cnt = Counter((x2 - x1, y2 - y1) for x1, y1 in a for x2, y2 in b)
        return max(cnt.values(), default=0)
```

**Checklist trước khi submit:**

- [ ] Xét **mọi hướng** dịch (cả âm lẫn dương) và cả hai trục?
- [ ] Bit trượt ra ngoài bị **xoá** (không dịch vòng)?
- [ ] Chỉ đếm vị trí **cả hai đều là `1`** (không đếm `0-0`)?
- [ ] Dùng **một** hướng vector nhất quán (`b − a`) cho mọi cặp?
- [ ] `max(..., default=0)` khi một ảnh không có ô `1`?
- [ ] Đã thử `[[1]],[[1]]` (1), `[[0]],[[0]]` (0), ví dụ 1 (3), hai ô ở hai góc đối diện?
- [ ] Đúng tên `class Solution`, `largestOverlap(self, img1, img2)`, không in debug.